<a href="https://colab.research.google.com/github/BadrKandri/Sofac-commercial-agent/blob/main/Sofac_extractor_model_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1-Setup

In [1]:
from google.colab import drive

!fusermount -u /gdrive
drive.mount('/gdrive', force_remount=True)

Mounted at /gdrive


In [ ]:
!pip install -qU transformers[torch] peft accelerate bitsandbytes
!pip install -qU sentence-transformers
!pip install -qU uvicorn pyngrok nest-asyncio pydantic starlette

In [27]:
import os, re, json, torch, requests, threading, uvicorn, nest_asyncio, asyncio
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from typing import List, Dict, Tuple, Any, Optional, TypedDict, Literal, Union
import wandb, os, json, random, torch, requests, threading
from fastapi.middleware.cors import CORSMiddleware
from google.colab import drive, userdata
from pydantic import BaseModel, Field
from huggingface_hub import login
from datetime import datetime
from tqdm.auto import tqdm
from peft import PeftModel
from pyngrok import ngrok
from os.path import join

In [6]:
dir = "/gdrive/MyDrive/pfa_Sofac"
training_dir = dir + "/extractor_model_finetuning"
dossier_cache_drive = dir + "/LLM_Cache"

os.environ["WANDB_API_KEY"] = userdata.get('Wandb_Key')
wandb.login()

hf_token = userdata.get('HF-TOKEN')
login(token=hf_token)

device = "cuda" #gpu
torch_dtype = None

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: badr-kandri (badr-kandri-ensa-berrchid) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


#2- Load Base Model

In [7]:
base_model_id = "Qwen/Qwen2.5-7B-Instruct"

def get_model_and_tokenizer(model_id):
  model = AutoModelForCausalLM.from_pretrained(
  model_id,
  quantization_config=quantization_config,
  device_map="auto",
  dtype=torch_dtype,
  cache_dir=dossier_cache_drive)
  tokenizer = AutoTokenizer.from_pretrained(model_id)

  return model, tokenizer

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

base_model, tokenizer = get_model_and_tokenizer(base_model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

#3- Model Pipeline

In [8]:
ProfilClient = Literal["fonctionnaire", "salarie_prive", "profession_liberale", "artisan", "commercant", "retraite", "professionnel_entreprise"]
TypeProduit = Literal["credit_personnel", "credit_auto", "credit_auto_loa", "regroupement_credits", "credit_equipement", "leasing"]
TypeVehicule = Literal["voiture_neuve", "voiture_occasion", "deux_roues_neuf", "deux_roues_occasion", "vehicule_utilitaire"]
ObjetFinancement = Literal["voiture_neuve", "voiture_occasion", "deux_roues", "equipement_menager", "mobilier", "travaux_decoration", "projet_professionnel", "usage_libre"]

class InfoClient(BaseModel):
    profile_client: ProfilClient = Field(..., description= "Statut professionnel de l'emprunteur")
    revenu_net_mensuel: int = Field(..., description= "Revenu net mensuel de l'emprunteur en DH")
    date_naissance: str = Field(...,
        description="Date de naissance au format jj/mm/aaaa",
        pattern=r"^(0[1-9]|[12][0-9]|3[01])/(0[1-9]|1[0-2])/\d{4}$")

    cin: str = Field(..., description= "Numéro d'identification de l'emprunteur")
    telephone: str = Field(..., description= "Numéro de téléphone de l'emprunteur")
    email: str = Field(..., description= "Adresse email de l'emprunteur")

class DetailVehicule(BaseModel):
    is_auto_loa : bool = Field(..., description= "True si type_produit est un credit_auto ou un credit_auto_loa, False sinon")
    prix_vehicule : int = Field(..., description= "Prix d'achat total du véhicule en DH.")
    apport_client : int = Field(..., description= "Montant de l'apport du client en DH, le client peux refuser d'apporter de l'argent la valeur est donc 0")
    type_vehicule : TypeVehicule = Field(..., description= "Catégorie du véhicule ciblé par l'emprunteur")

class InfoFinancement(BaseModel):
    type_produit: TypeProduit = Field(..., description="Le type de crédit souhaité...")
    details_vehicule: DetailVehicule = Field(..., description="Les détails concernant le véhicule souhaité par le prospect")
    mensualite_souhaitee: int = Field(...)
    duree_souhaitee: int = Field(...)
    montant_souhaite: Optional[int] = Field(default=None, description="...")

class Eligibility(BaseModel):
    objet_financement: ObjetFinancement = Field(..., description="L'objet de financement souhaité par le prospect parmi l'offre CREDIZ")
    charges_credits_en_cours: int = Field(..., description="Montant total des mensualités de crédits déjà en cours chez CREDIZ ou d'autres établissements en DH. Le client peut ne pas avoir de crédit en cours ou refuser de préciser le montant, la valeur est alors 0.")
    nombre_credits_en_cours: int = Field(..., description="Nombre de crédits actifs chez CREDIZ ou d'autres établissements. Le client peut ne pas avoir de crédit en cours ou refuser de répondre, la valeur est alors 0.")
    montant_a_racheter: int = Field(..., description="Capital restant dû sur les crédits à regrouper ou à racheter en DH. Le client peut refuser de préciser ce montant, la valeur est alors 0.")

# Wrapper
class DossierCredit(BaseModel):
    etat_qualification: Literal["incomplet", "complet"] = Field(..., description="Passe à 'complet' UNIQUEMENT lorsque toutes les informations requises pour le type de crédit souhaité ont été récoltées.")
    info_client: Optional[InfoClient] = Field(default=None, description="Les informations personnelles du prospect.")
    info_financement: Optional[InfoFinancement] = Field(default=None, description="Les détails financiers du crédit demandé.")
    eligibilite: Optional[Eligibility] = Field(default=None, description="Les informations liées à la situation financière actuelle du prospect.")

In [9]:
# user_msg -> json -> template text -> tokens -> pt tensors -> next token -> human response -> json
def load_message(message_client):

  schema = DossierCredit.model_json_schema()
  schema_extraction_sofac = [
      {
          "role": "system",
          "content": "\n".join([
              "Tu es un expert financier et un système d'extraction de données pour SOFAC.",
              "Ton objectif est d'extraire les informations du prospect et de remplir LE PLUS POSSIBLE les sous-objets presents dans la variable need et remplire le JSON.",
              "CRITIQUE : Ne renvoie 'null' pour un sous-objet QUE SI absolument aucune information du texte ne s'y rapporte. Dès qu'un détail existe (ex: métier, montant, voiture), tu DOIS créer le sous-objet et le remplir.",
              "Tu ne dois générer AUCUN texte conversationnel, ni introduction, ni conclusion.",
              "Réponds uniquement avec un objet JSON valide."
          ])
      },
      {
          "role": "user",
          "content": "\n".join([
              "## Message du Client :",
              message_client.strip(),
              "",
              "## Schéma JSON d'extraction attendu :",
              json.dumps(schema, ensure_ascii=False),
              "",
              "## Données Extraites :",
              "```json\n"
          ])
      }
  ]

  return schema_extraction_sofac

def generate_response(messages, model, tokenizer):
    # 1 - The tokenizer transforms the json into text that the model was trained to understand using its template
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # 2 - The tokenizer transforms the text into tokens, then into PyTorch tensors (numbers), and sends them to the active hardware device that contains the model (GPU T4)
    inputs = tokenizer([text], return_tensors="pt").to(device)

    # 3 - Use the input tokens to predict the next tokens: 'generate()' function returns a sequence containing the user prompt IDs and the model output IDs
    generated_ids = model.generate(
        inputs.input_ids,
        max_new_tokens=1024, # limit the maximum tokens the model can generate
        do_sample=False,
        top_k=None,
        temperature=None,
        top_p=None)

    # 4 - Remove the prompt IDs and leave just the model output IDs
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]

    # 5 - Generate response: Decode the model output IDs into human text
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

def parse_and_validate(response):
  # Transform the response into a python dictionary
  try:
      dossier_extrait = json.loads(response)
      return json.dumps(dossier_extrait, indent=4, ensure_ascii=False)
  except json.JSONDecodeError:
      return "The model failed to generate a valide JSON. This is the response : \n" + response


In [10]:
def extractor_pipeline(message_client, model, tokenizer):
    messages = load_message(message_client)
    raw_response = generate_response(messages, model, tokenizer)
    final_data = parse_and_validate(raw_response)
    return final_data

#4- Knowledge Distilation

In [ ]:
"""
================================================================================
SOFAC / CREDIZ — Générateur de Dataset Synthétique pour Fine-Tuning (SFT)
================================================================================
Architecture : Knowledge Distillation via API Groq (Llama-3 comme "Professeur")
Cible        : Fine-tuning supervisé du modèle Qwen-4B (Extracteur de slots)
Format sortie : ChatML → fichier .jsonl
Environnement : Google Colab (clé API via userdata.get)

Auteur  : Lead Data Engineer SOFAC
Version : 1.3.0 — Key Pooling (Rotation Automatique sur 3 Clés)
================================================================================
"""

import json
import time
import random
import logging
import os
import re
from datetime import datetime, UTC
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

from google.colab import userdata
from groq import Groq, APIConnectionError, APIStatusError, RateLimitError

# ─────────────────────────────────────────────────────────────────────────────
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║             PANNEAU DE CONTRÔLE — MODIFIER ICI UNIQUEMENT              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ─────────────────────────────────────────────────────────────────────────────

# 🎯 Nombre total d'exemples à générer par exécution du script
TARGET_SAMPLES      = 500

# 🔑 Liste des secrets Colab contenant vos clés API Groq
#    Le script basculera automatiquement à la suivante si la précédente est épuisée.
COLAB_SECRET_NAMES  = ["groq-key", "groq-key2", "groq-key3", "groq-key4","groq-key5"]

# 🤖 Modèle Groq à utiliser comme "Professeur"
GROQ_MODEL          = "llama-3.3-70b-versatile"

# ⚙️ Paramètres de génération
MAX_TOKENS          = 1200
TEMPERATURE         = 0.95    # Haute créativité pour simuler le chaos client
RETRY_MAX           = 3       # Tentatives pour erreurs serveur (500, etc.)
RETRY_DELAY_SEC     = 5
DELAY_BETWEEN_CALLS = 12.0    # 12s entre appels = ~5 req/min par clé

# 📁 Répertoire de sortie
OUTPUT_DIR          = Path("/gdrive/MyDrive/pfa_Sofac/llm-finetuning/dataset_sofac")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION DÉRIVÉE (ne pas modifier)
# ─────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("SOFAC-DataGen")

# ── Chargement et Rotation des Clés API ──────────────────────────────────────
GROQ_CLIENTS = []
for secret_name in COLAB_SECRET_NAMES:
    try:
        key = userdata.get(secret_name)
        if key:
            GROQ_CLIENTS.append(Groq(api_key=key))
            logger.info(f"✅ Clé API chargée avec succès : '{secret_name}'")
    except Exception:
        logger.warning(f"⚠️ Impossible de charger le secret '{secret_name}'")

if not GROQ_CLIENTS:
    logger.error("❌ FATAL : Aucune clé API Groq n'a pu être chargée.")
    raise RuntimeError("Toutes les clés API sont manquantes.")

# ── Chemin de sortie ────────────────────────────────────────────────────────
OUTPUT_FILE = OUTPUT_DIR / "sofac_sft_dataset.jsonl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exception personnalisée pour la rotation
class GroqRateLimitReached(Exception):
    pass

# ─────────────────────────────────────────────────────────────────────────────
# MATRICE DE SCÉNARIOS
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class ScenarioDimension:
    langue: str
    chaos_level: str
    profil_client: str
    type_produit: str
    completude: str

LANGUES = {
    "francais_pur": "Le client écrit en français standard, phrases complètes et correctes.",
    "darija_arabe": "Le client écrit exclusivement en Darija marocaine, alphabet arabe.",
    "arabizi_franco": "Le client utilise le Franco-Arabe (Arabizi) : Darija écrite en lettres latines.",
    "melange_total": "Le client mélange aléatoirement français, darija arabe et arabizi dans la même phrase.",
}

CHAOS_LEVELS = {
    "client_bavard": "Le client raconte sa vie avant de donner une info utile. Message long.",
    "client_fantome": "Le client donne une info minimale et vague (ex: 'salam', 'bghit flous').",
    "client_hors_sujet": "Le client est hors sujet : plaintes, adresse agence, météo.",
    "client_partiel": "Le client donne 2 ou 3 informations utiles mais il en manque d'autres.",
    "client_complet": "Le client donne exceptionnellement TOUTES les infos nécessaires d'un coup.",
}

PROFILS_CLIENT = [
    "fonctionnaire", "salarie_prive", "profession_liberale",
    "artisan", "commercant", "retraite", "professionnel_entreprise"
]

TYPES_PRODUIT = [
    "credit_personnel", "credit_auto", "credit_auto_loa",
    "regroupement_credits", "credit_equipement", "leasing"
]

def build_scenario_matrix() -> list[ScenarioDimension]:
    matrix = []
    for langue_key in LANGUES:
        for chaos_key in CHAOS_LEVELS:
            for produit in TYPES_PRODUIT:
                profil = random.choice(PROFILS_CLIENT)
                completude = "complet" if chaos_key == "client_complet" else "incomplet"
                matrix.append(
                    ScenarioDimension(
                        langue=langue_key, chaos_level=chaos_key, profil_client=profil,
                        type_produit=produit, completude=completude
                    )
                )
    return matrix

# ─────────────────────────────────────────────────────────────────────────────
# PROMPT PROFESSEUR (TEACHER PROMPT)
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT_TEACHER = """
Tu es un générateur de données d'entraînement spécialisé pour un agent IA de crédit marocain (SOFAC / CREDIZ).
Ta mission est de produire des paires (message_utilisateur, json_extraction) réalistes et variées.

RÈGLE 1 — FORMAT DE SORTIE
Réponds UNIQUEMENT avec un objet JSON valide.

RÈGLE 2 — STRUCTURE IMPOSÉE
{
  "user_message": "<string>",
  "assistant_json": {
    "etat_qualification": "<'incomplet' | 'complet'>",
    "info_client": {
      "profile_client": "<valeur Literal ou null>",
      "revenu_net_mensuel": <entier ou null>,
      "date_naissance": "<ISO 8601 ou null>",
      "cin": "<string ou null>",
      "telephone": "<string ou null>",
      "email": "<string ou null>"
    },
    "info_financement": {
      "type_produit": "<valeur Literal ou null>",
      "details_vehicule": {
        "is_auto_loa": <bool ou null>,
        "prix_vehicule": <entier ou null>,
        "apport_client": <entier ou null>,
        "type_vehicule": "<valeur Literal ou null>"
      },
      "montant_souhaite": <entier ou null>,
      "duree_souhaitee": <entier ou null>,
      "mensualite_souhaitee": <entier ou null>
    },
    "eligibilite": {
      "objet_financement": "<valeur Literal ou null>",
      "charges_credits_en_cours": <entier ou null>,
      "nombre_credits_en_cours": <entier ou null>,
      "montant_a_racheter": <entier ou null>
    }
  }
}

RÈGLE 3 — VALEURS AUTORISÉES (Literals)
N'utilise JAMAIS une valeur hors de ces listes :
profile_client       : "fonctionnaire" | "salarie_prive" | "profession_liberale" | "artisan" | "commercant" | "retraite" | "professionnel_entreprise"
type_produit         : "credit_personnel" | "credit_auto" | "credit_auto_loa" | "regroupement_credits" | "credit_equipement" | "leasing"
type_vehicule        : "voiture_neuve" | "voiture_occasion" | "deux_roues_neuf" | "deux_roues_occasion" | "vehicule_utilitaire"
objet_financement    : "voiture_neuve" | "voiture_occasion" | "deux_roues" | "equipement_menager" | "mobilier" | "travaux_decoration" | "projet_professionnel" | "usage_libre"
etat_qualification   : "incomplet" | "complet"

RÈGLE 4 — NULL ET NON ZÉRO
Si une information n'est PAS présente dans le message utilisateur, utilise OBLIGATOIREMENT null. Jamais 0, jamais "".
"""

def build_user_prompt(scenario: ScenarioDimension) -> str:
    return f"""
Génère UN exemple de conversation selon ces paramètres :
Langue/Registre : {scenario.langue} ({LANGUES[scenario.langue]})
Niveau chaos    : {scenario.chaos_level} ({CHAOS_LEVELS[scenario.chaos_level]})
Profil client   : {scenario.profil_client}
Produit CREDIZ  : {scenario.type_produit}
Complétude      : {scenario.completude}

Invente un message authentique et marocain. L'extraction JSON doit refléter STRICTEMENT ce message.
"""

# ─────────────────────────────────────────────────────────────────────────────
# CLIENT API GROQ
# ─────────────────────────────────────────────────────────────────────────────

def call_groq_api(active_client: Groq, user_prompt: str) -> Optional[dict]:
    for attempt in range(1, RETRY_MAX + 1):
        try:
            response = active_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT_TEACHER},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                response_format={"type": "json_object"},
            )
            raw_content = response.choices[0].message.content.strip()
            raw_content = re.sub(r"^```(?:json)?\s*", "", raw_content)
            raw_content = re.sub(r"\s*```$", "", raw_content)
            return json.loads(raw_content)

        except RateLimitError:
            # ERREUR 429 : On lève l'exception pour déclencher la rotation dans le main loop
            raise GroqRateLimitReached()

        except Exception as e:
            logger.warning(f"[Tentative {attempt}/{RETRY_MAX}] Erreur API non bloquante : {e}")
            time.sleep(RETRY_DELAY_SEC)

    return None

# ─────────────────────────────────────────────────────────────────────────────
# FORMATEUR CHATML → JSONL
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT_EXTRACTEUR = """
Tu es l'extracteur de données de l'agent CREDIZ (SOFAC Maroc).
Ton rôle est UNIQUE : lire le message du client et extraire les informations dans un JSON strict.
Tu ne parles pas au client. Tu ne poses pas de questions. Tu extrais et tu retournes uniquement le JSON.

Valeurs autorisées :
- profile_client       : fonctionnaire | salarie_prive | profession_liberale | artisan | commercant | retraite | professionnel_entreprise
- type_produit         : credit_personnel | credit_auto | credit_auto_loa | regroupement_credits | credit_equipement | leasing
- type_vehicule        : voiture_neuve | voiture_occasion | deux_roues_neuf | deux_roues_occasion | vehicule_utilitaire
- objet_financement    : voiture_neuve | voiture_occasion | deux_roues | equipement_menager | mobilier | travaux_decoration | projet_professionnel | usage_libre
- etat_qualification   : incomplet | complet

Règle : toute information absente = null (jamais 0, jamais "").
"""

def format_as_chatml(raw_sample: dict, scenario: ScenarioDimension) -> Optional[dict]:
    try:
        user_message   = raw_sample["user_message"]
        assistant_json = raw_sample["assistant_json"]
        assistant_content = json.dumps(assistant_json, ensure_ascii=False, separators=(",", ":"))

        return {
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT_EXTRACTEUR},
                {"role": "user",      "content": user_message},
                {"role": "assistant", "content": assistant_content},
            ],
            "metadata": {
                "langue":        scenario.langue,
                "chaos_level":   scenario.chaos_level,
                "profil_client": scenario.profil_client,
                "type_produit":  scenario.type_produit,
                "completude":    scenario.completude,
                "generated_at":  datetime.now(UTC).isoformat(),
            },
        }
    except Exception as e:
        logger.error(f"Erreur de formatage ChatML : {e}")
        return None

# ─────────────────────────────────────────────────────────────────────────────
# PIPELINE PRINCIPAL (Propre et sans erreur)
# ─────────────────────────────────────────────────────────────────────────────

def run_generation_pipeline():
    logger.info("=" * 70)
    logger.info("SOFAC / CREDIZ — Démarrage du pipeline de génération")
    logger.info(f"Pool de clés API : {len(GROQ_CLIENTS)} clé(s) actives")
    logger.info(f"Fichier cible    : {OUTPUT_FILE}")
    logger.info("=" * 70)

    matrix = build_scenario_matrix()

    # Echantillonnage
    if TARGET_SAMPLES <= len(matrix):
        all_scenarios = random.sample(matrix, TARGET_SAMPLES)
    else:
        repeats = (TARGET_SAMPLES // len(matrix)) + 1
        expanded = matrix * repeats
        for s in expanded:
            s.profil_client = random.choice(PROFILS_CLIENT)
        all_scenarios = random.sample(expanded, TARGET_SAMPLES)

    random.shuffle(all_scenarios)

    success_count = 0
    current_client_idx = 0

    # MODE APPEND 'a' : Ajoute au fichier existant
    with open(OUTPUT_FILE, "a", encoding="utf-8") as out_f:
        for idx, scenario in enumerate(all_scenarios, start=1):

            user_prompt = build_user_prompt(scenario)
            raw_sample = None

            while current_client_idx < len(GROQ_CLIENTS):
                active_client = GROQ_CLIENTS[current_client_idx]
                logger.info(f"[{idx}/{TARGET_SAMPLES}] Génération avec clé {current_client_idx + 1} | {scenario.langue} | {scenario.chaos_level}")

                try:
                    raw_sample = call_groq_api(active_client, user_prompt)
                    break
                except GroqRateLimitReached:
                    logger.warning(f"⚠️ Quota épuisé pour la clé {current_client_idx + 1}.")
                    current_client_idx += 1
                    if current_client_idx < len(GROQ_CLIENTS):
                        logger.info(f"🔄 Basculement sur la clé {current_client_idx + 1}...")

            if current_client_idx >= len(GROQ_CLIENTS):
                logger.error("🛑 Toutes les clés API Groq sont épuisées pour aujourd'hui.")
                break

            if raw_sample:
                chatml_entry = format_as_chatml(raw_sample, scenario)
                if chatml_entry:
                    out_f.write(json.dumps(chatml_entry, ensure_ascii=False) + "\n")
                    out_f.flush()
                    success_count += 1

            time.sleep(DELAY_BETWEEN_CALLS)

    # ── MESSAGE DE SYNTHÈSE FINAL ────────────────────────────────────────────
    # Calcul du total actuel de lignes dans le fichier
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        total_data_so_far = sum(1 for _ in f)

    logger.info("=" * 70)
    logger.info("✅ SYNTHÈSE DE LA GÉNÉRATION")
    logger.info(f"   ➤ Data générée dans cette tentative : {success_count} exemples")
    logger.info(f"   ➤ Total cumulé de data dans le fichier : {total_data_so_far} lignes")
    logger.info("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# POINT D'ENTRÉE
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    run_generation_pipeline()

#5- Finetuning

##5.1- Format Datasets

In [11]:
data_set= training_dir + "/dataset"
data_path = data_set + "/sofac_sft_dataset.jsonl"
llm_finetunning_data = []


system_message =  "\n".join([
    "You are a professional NLP data parser. Extract information from the message ",
    "according to the output scheme. Output ONLY valid JSON."
])

with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip() == "": continue
        rec = json.loads(line.strip())

        user_msg = rec['messages'][1]['content']
        assistant_json = json.loads(rec['messages'][2]['content'])

        llm_finetunning_data.append({
            "system": system_message,
            "instruction": "Extraire les informations du message client vers le schéma JSON.",
            "input": user_msg,
            "output": json.dumps(assistant_json, ensure_ascii=False),
            "history": []
        })

# Mélange des données pour l'entraînement
random.Random(101).shuffle(llm_finetunning_data)

In [12]:
len(llm_finetunning_data)

598

In [13]:
train_data_size = 550
evaluation_data_size = 48


train_data = llm_finetunning_data[:train_data_size]
evaluation_data = llm_finetunning_data[train_data_size:]

with open(join(data_set, "train_set.json"), "w") as dest:
    json.dump(train_data, dest, ensure_ascii=False, default=str)

with open(join(data_set, "evaluation_set.json"), "w", encoding="utf8") as dest:
    json.dump(evaluation_data, dest, ensure_ascii=False, default=str)


In [14]:
print(join(data_set, "train_set.json"))
print(join(data_set, "evaluation_set.json"))

/gdrive/MyDrive/pfa_Sofac/extractor_model_finetuning/dataset/train_set.json
/gdrive/MyDrive/pfa_Sofac/extractor_model_finetuning/dataset/evaluation_set.json


In [ ]:
'''
"extractor_finetune_train": {
        "file_name": "/gdrive/MyDrive/pfa_Sofac/extractor_model_finetuning/dataset/train_set.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "extractor_finetune_val": {
        "file_name": "/gdrive/MyDrive/pfa_Sofac/extractor_model_finetuning/dataset/evaluation_set.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }
'''

##5.2- Yaml file configuration

In [ ]:
%%writefile /content/LlamaFactory/examples/train_lora/sofac_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-7B-Instruct
quantization_bit: 4
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_target: all

### dataset
dataset: extractor_finetune_train
eval_dataset: extractor_finetune_val
template: qwen
cutoff_len: 1024
# max_samples: 500
overwrite_cache: true
preprocessing_num_workers: 4

### output
# resume_from_checkpoint: /gdrive/MyDrive/pfa_Sofac/extractor_model_finetuning/model/checkpoint-100
output_dir: /gdrive/MyDrive/pfa_Sofac/extractor_model_finetuning/model/
logging_steps: 10
save_steps: 25
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 16 # max_samples/16 = nbr_etape par époque
learning_rate: 1.0e-4
num_train_epochs: 3.0 # steps = nbr_etape total x 3
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

report_to: wandb
run_name: sofac-finetune-llamafactory

push_to_hub: true
export_hub_model_id: "Meliodas-10/Qwen_output"
hub_private_repo: true
hub_strategy: checkpoint


##5.3- Run Finetuning

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!cd LlamaFactory/ && llamafactory-cli train /content/LlamaFactory/examples/train_lora/sofac_finetune.yaml

#6- Evaluation

In [ ]:
finetuned_model_id = "Meliodas-10/Sofac_extractor_model"
extractor_model = PeftModel.from_pretrained(base_model, finetuned_model_id)

In [16]:
message_client = "Bonjour, je suis Badr je fabrique des portes et je veux acheter une voiture d'occasion à 200 000 DH et je suis nee en janvier 2004"
result = extractor_pipeline(message_client, base_model, tokenizer)
print(result)

{
    "etat_qualification": "incomplet",
    "info_client": {
        "profile_client": "artisan",
        "revenu_net_mensuel": null,
        "date_naissance": "01/01/2004",
        "cin": null,
        "telephone": null,
        "email": null
    },
    "info_financement": {
        "type_produit": "credit_auto",
        "details_vehicule": {
            "is_auto_loa": false,
            "prix_vehicule": 200000,
            "apport_client": null,
            "type_vehicule": "voiture_occasion"
        },
        "mensualite_souhaitee": null,
        "duree_souhaitee": null
    },
    "eligibilite": {
        "objet_financement": null,
        "charges_credits_en_cours": null,
        "nombre_credits_en_cours": null,
        "montant_a_racheter": null
    }
}


#7- Re-Finetuning

In [ ]:
'''
  "sofac_extractor_finetune_train": {
    "file_name": "/content/drive/MyDrive/extractor_model_finetuning/dataset/sofac_extractor_finetune_train.json",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output",
      "system": "system",
      "history": "history"
    }
  },
  "sofac_extractor_finetune_val": {
    "file_name": "/content/drive/MyDrive/extractor_model_finetuning/dataset/sofac_extractor_finetune_eval.json",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output",
      "system": "system",
      "history": "history"
    }
  }
'''

##7.1- Yaml file configuration

In [ ]:
%%writefile /content/LlamaFactory/examples/train_lora/sofac_extractor_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-7B-Instruct
quantization_bit: 4
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_target: all

### dataset
dataset: sofac_extractor_finetune_train
eval_dataset: sofac_extractor_finetune_val
template: qwen
cutoff_len: 512
# max_samples: 500
overwrite_cache: true
preprocessing_num_workers: 4

### output
resume_from_checkpoint: /content/drive/MyDrive/extractor_model_finetuning/Sofac_extractor_model/checkpoint-350
output_dir: /content/drive/MyDrive/extractor_model_finetuning/Sofac_extractor_model/
logging_steps: 50
save_steps: 50
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 10 # max_samples/10 = nbr_etape par époque
learning_rate: 1.0e-4
num_train_epochs: 3.0 # steps = nbr_etape total x 3
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 40 #run evaluation every 50 steps (calculate loss)

report_to: wandb
run_name: sofac_extractor_model-finetune-llamafactory

push_to_hub: true
export_hub_model_id: "Meliodas-10/sofac_extractor-model"
hub_private_repo: true
hub_strategy: checkpoint


##7.2- Run Finetuning

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!cd LlamaFactory/ && llamafactory-cli train /content/LlamaFactory/examples/train_lora/sofac_extractor_finetune.yaml

#8- New Pipeline

In [30]:
ProfilClient = Literal["fonctionnaire", "salarie_prive", "profession_liberale", "artisan", "commercant", "retraite", "professionnel_entreprise"]
TypeProduit = Literal["credit_personnel", "credit_auto", "credit_auto_loa", "regroupement_credits", "credit_equipement", "leasing"]
TypeVehicule = Literal["voiture_neuve", "voiture_occasion", "deux_roues_neuf", "deux_roues_occasion", "vehicule_utilitaire"]
ObjetFinancement = Literal["voiture_neuve", "voiture_occasion", "deux_roues", "equipement_menager", "mobilier", "travaux_decoration", "projet_professionnel", "usage_libre"]

class InfoClient(BaseModel):
    profile_client: ProfilClient = Field(..., description= "Statut professionnel de l'emprunteur")
    revenu_net_mensuel: int = Field(..., description= "Revenu net mensuel de l'emprunteur en DH")
    date_naissance: str = Field(...,
        description="Date de naissance au format jj/mm/aaaa",
        pattern=r"^(0[1-9]|[12][0-9]|3[01])/(0[1-9]|1[0-2])/\d{4}$"
    )
    cin: str = Field(..., description= "Numéro d'identification de l'emprunteur")
    telephone: str = Field(..., description= "Numéro de téléphone de l'emprunteur")
    email: str = Field(..., description= "Adresse email de l'emprunteur")

class DetailVehicule(BaseModel):
    is_auto_loa : bool = Field(..., description= "True si type_produit est un credit_auto ou un credit_auto_loa, False sinon")
    prix_vehicule : int = Field(..., description= "Prix d'achat total du véhicule en DH.")
    apport_client : int = Field(..., description= "Montant de l'apport du client en DH, le client peux refuser d'apporter de l'argent la valeur est donc 0")
    type_vehicule : TypeVehicule = Field(..., description= "Catégorie du véhicule ciblé par l'emprunteur")

class InfoFinancement(BaseModel):
    type_produit: TypeProduit = Field(..., description="Le type de crédit souhaité...")
    details_vehicule: DetailVehicule = Field(..., description="Les détails concernant le véhicule souhaité par le prospect")
    mensualite_souhaitee: int = Field(...)
    duree_souhaitee: int = Field(...)
    montant_souhaite: Optional[int] = Field(default=None, description="...")

class Eligibility(BaseModel):
    objet_financement: ObjetFinancement = Field(..., description="L'objet de financement souhaité par le prospect parmi l'offre CREDIZ")
    charges_credits_en_cours: int = Field(..., description="Montant total des mensualités de crédits déjà en cours chez CREDIZ ou d'autres établissements en DH. Le client peut ne pas avoir de crédit en cours ou refuser de préciser le montant, la valeur est alors 0.")
    nombre_credits_en_cours: int = Field(..., description="Nombre de crédits actifs chez CREDIZ ou d'autres établissements. Le client peut ne pas avoir de crédit en cours ou refuser de répondre, la valeur est alors 0.")
    montant_a_racheter: int = Field(..., description="Capital restant dû sur les crédits à regrouper ou à racheter en DH. Le client peut refuser de préciser ce montant, la valeur est alors 0.")

# Wrapper
class DossierCredit(BaseModel):
    etat_qualification: Literal["incomplet", "complet"] = Field(..., description="Passe à 'complet' UNIQUEMENT lorsque toutes les informations requises pour le type de crédit souhaité ont été récoltées.")
    info_client: Optional[InfoClient] = Field(default=None, description="Les informations personnelles du prospect.")
    info_financement: Optional[InfoFinancement] = Field(default=None, description="Les détails financiers du crédit demandé.")
    eligibilite: Optional[Eligibility] = Field(default=None, description="Les informations liées à la situation financière actuelle du prospect.")

In [31]:
 def load_message(message_client, schema, needs: list[str], demande):
    needs_str = ", ".join(needs) if needs else "AUCUN"

    schema_extraction_sofac = [
        {
            "role": "system",
            "content": "\n".join([
                "Tu es un algorithme d'extraction de données ultra-strict pour SOFAC.",
                "",
                "RÈGLE 1 : EXTRACTION RESTREINTE (TRÈS IMPORTANT)",
                f"Tu es autorisé à extraire UNIQUEMENT ces champs : [{needs_str}].",
                "Tous les autres champs du schéma DOIVENT OBLIGATOIREMENT être 'null', même si le client donne l'information.",
                "",
                "RÈGLE 2 : ZÉRO INVENTION (ANTI-HALLUCINATION)",
                "Ne devine JAMAIS une valeur. Le client n'a pas donné de revenu exact ? Alors revenu_net_mensuel = null.",
                "",
                "RÈGLE 3 : REFUS EXPLICITE OU ABSENCE = 0, PAS NULL (pour les champs numériques concernés)",
                "Pour les champs de type montant/nombre où une absence RÉELLE est une information valide",
                "(apport_client, charges_credits_en_cours, nombre_credits_en_cours, montant_a_racheter),",
                "si le client dit explicitement qu'il n'en a pas, refuse d'en donner, ou répond '0', '0dh', '0 DH', 'non', 'aucun',",
                "alors la valeur du champ est 0 (nombre entier), PAS null.",
                "null signifie 'je ne sais pas / pas mentionné'. 0 signifie 'le client a confirmé qu'il n'y en a pas'.",
                "",
                "## EXEMPLES",
                "Question de l'agent : 'Avez-vous des crédits en cours ?'",
                "Message client : 'non je n'ai pas de charges en cours'",
                "Besoins autorisés : ['charges_credits_en_cours', 'nombre_credits_en_cours']",
                "Action : charges_credits_en_cours=0, nombre_credits_en_cours=0 (refus explicite, pas absence d'info).",
                "",
                "Question de l'agent : 'Veuillez me donner votre CIN.'",
                "Message client : 'Je suis médecin, je veux un crédit de 50000 DH. Mon CIN est AB1234.'",
                "Besoins autorisés : ['cin']",
                "Action : cin='AB1234'. Le montant (50000) et la profession (médecin) restent 'null' car hors besoins autorisés — ceci n'est PAS un cas de refus, juste hors périmètre.",
                "",
                "Réponds uniquement avec l'objet JSON attendu."
            ])
        },
        {
            "role": "user",
            "content": "\n".join([
                "## Question posée par l'agent :",
                demande,
                "",
                "## Réponse du Client :",
                message_client.strip(),
                "",
                "## Consigne d'extraction :",
                f"Cherche UNIQUEMENT les valeurs pour : [{needs_str}].",
                "Ignore le reste et force absolument TOUS les autres champs à null.",
                "",
                "## Schéma JSON :",
                json.dumps(schema, ensure_ascii=False),
                "",
                "## JSON Extrait :",
                "```json\n"
            ])
        }
    ]
    return schema_extraction_sofac

def encoder(message, model, tokenizer):
    text = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    return inputs

def output_generator(inputs, model):
    generated_ids = model.generate(
        inputs.input_ids,
        max_new_tokens=512,
        do_sample=False
    )

    # Strip prompt IDs from the output sequence
    isolated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]

    return isolated_ids

def decoder(isolated_ids, tokenizer):
    response = tokenizer.batch_decode(isolated_ids, skip_special_tokens=True)[0]
    return response.strip()

def parse_and_validate(response):
  try:
      dossier_extrait = json.loads(response)
      return json.dumps(dossier_extrait, indent=4, ensure_ascii=False)
  except json.JSONDecodeError:
      return "The model failed to generate a valide JSON. This is the response : \n" + response


In [32]:
def extractor_pipeline(query, needs, demande, model, tokenizer):
  schema = DossierCredit.model_json_schema()
  message = load_message(query, schema, needs, demande)
  tensor_inputs = encoder(message, model, tokenizer)
  generated_tokens = output_generator(tensor_inputs, model)
  answer = decoder(generated_tokens, tokenizer)
  final_answer = parse_and_validate(answer)

  return final_answer

#9- Test Model

In [35]:
needs= ['profile_client', 'date_naissance', 'type_produit']
demande = "Veuillez me preciser votre profile personnel et votre date de naissance"
query = "Bonjour, je suis Badr je fabrique des portes et je veux un credit pour acheter une voiture d'occasion et je suis nee en janvier 2004"

data = extractor_pipeline(query, needs, demande, base_model, tokenizer)
print(data)

{
    "etat_qualification": "incomplet",
    "info_client": {
        "profile_client": "artisan",
        "revenu_net_mensuel": null,
        "date_naissance": "01/01/2004",
        "cin": null,
        "telephone": null,
        "email": null
    },
    "info_financement": {
        "type_produit": "credit_auto",
        "details_vehicule": {
            "is_auto_loa": null,
            "prix_vehicule": null,
            "apport_client": null,
            "type_vehicule": null
        },
        "mensualite_souhaitee": null,
        "duree_souhaitee": null
    },
    "eligibilite": {
        "objet_financement": null,
        "charges_credits_en_cours": null,
        "nombre_credits_en_cours": null,
        "montant_a_racheter": null
    }
}


#END